In [1]:
import torch.nn as nn
import torch 
import esm

device = "mps"


In [ ]:



class ESMWrapper(nn.Module):
    """Wrapper um Denoiser: Aufbau eines Embeddingraumes und decoding in Aminosäurelänge
    ### Args:
        - device (string, wird im Geräte unabhängigen Code definiert.), "cpu" als default.
    """
    def __init__(self, device: str = "cpu", L_max: int= 42, *args, **kwargs):
        super().__init__(*args, **kwargs)
        model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
        batch_converter = alphabet.get_batch_converter()
        AS = "LAGVSERTIDPKQNFYMHWC"

        self.AS_list = [alphabet.tok_to_idx[a] for a in AS]
        self.model = model
        self.alphabet = alphabet
        self.batch_converter = batch_converter
        self.device = device
        self.L_max = L_max
        

    @torch.no_grad() #Protokoll für Backprop ausschalten
    def ESM_encode(self, seqs: list[str]) -> torch.Tensor: 
        """
        ### Encoder.
            Baut aus einem Batch von Peptidsequenzen ein ESM-2 Embedding. BOS und EOS werden abgeschnitten, 
            sodass Ausgabe mit Shape (B,L_batch, D) ausgegeben wird. 

        ### Args:
        - seqs als Inputliste mit Sequenzen, z.B ["ACDEFGHIKLMN", "KLVFFAED"]
        """
        data = [(f"seq{i}", s) for i, s in enumerate(seqs)]
        model = self.model
        seqs_lenghts = torch.tensor([len(s) for s in seqs]).unsqueeze(1) #dims (B) jeweilige AS-Sequenzlängen

        model.eval().to(self.device)
        _, _,seq_tokens = self.batch_converter(data) # dims (B,L_batch), mit PADs zum Auffüllen
        out = model(seq_tokens.to(self.device), repr_layers = [model.num_layers]) #dims (B)
        reps = out["representations"][model.num_layers]
        reps_sliced = reps[:,1:-1,:] #Abschneiden von BOS und EOS

        #==Padding mit torch.zeros()==
        positions = torch.arange(self.L_max).unsqueeze(0)
        padding_mask = positions < seqs_lenghts

        reps_B = reps_sliced.shape[0]
        reps_dims = reps_sliced.shape[2]
        reps_len = reps_sliced.shape[1]
        residues = self.L_max - reps_len
        resiues_tensor = torch.zeros(size=(reps_B, residues, reps_dims)).to(self.device)
        reps_return = torch.cat(tensors=(reps_sliced, resiues_tensor), dim= 1) # Aufgefüllt mit batch_converter Padding und 0

        #==Paddingmaske mit booleans==
        padding_mask = torch.ones(size = (reps_B, self.L_max), dtype=torch.bool, device = self.device) #False für gepaddete Positionen
        padding_mask[:, reps_len:] = 0

        #==Ausgabe==
        return reps_return.cpu(), padding_mask.cpu() # dims: representations (B,L_max, dims), padding mask (B, L_max)

    @torch.no_grad()
    def ESM_decode(self, reps: torch.Tensor, padding_mask: torch.Tensor) -> list:
        
        AS = "LAGVSERTIDPKQNFYMHWC"
        AS_list = [self.alphabet.tok_to_idx[a] for a in AS]        
        
        self.model.eval().to(self.device)
        logits = self.model.lm_head(reps) # dims: (B, L_batch, 33)
        seq_tokens = torch.argmax(logits[:,:, 4:24], dim=-1)

        #input dims: B,L_max, dims und vielleicht eine Padding-Maske? (kann eine KI eine Paddingmaske mitlernen?)
        


SyntaxError: invalid syntax (2083690314.py, line 58)

In [55]:


pep_a = "ACDEFGHIKLMNRADFR"   
pep_b = "KLVFFAED"       # 8 AS
X = [pep_a, pep_b]
wrapper = ESMWrapper(device= device)

tens, mask = wrapper.ESM_encode(seqs= X)
print(tens.shape)

mask.shape

torch.Size([2, 42, 320])


torch.Size([2, 42])

In [52]:
x = torch.randint(low=0, high=10, size=(2, 10,3))



design_len = 5
B, T = x.size(0), x.size(1)
            
# Create a tensor of zeros with size [B, T]
padding_mask = torch.zeros((B, T), dtype=torch.bool, device = x.device)
            
# Set values to one after the design_len+1 position, but leave the last position as one
padding_mask[:, design_len+1:] = 1

padding_mask

tensor([[False, False, False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True,  True,  True]])